# Step 4 Extension: Annotate Remaining 250 + Human Labelling File

1. Annotate rows 251-500 with condition B (few-shot Ministral)
2. Generate a clean human labelling file for you and Robin

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import os
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, cohen_kappa_score, confusion_matrix
)
import importlib
import annotation_setup
importlib.reload(annotation_setup)
from annotation_setup import VALID_LABELS, instructions, reminder, parse_label, parse_explanation

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Model: {MODEL}")

Model: ministral-3-14b


In [4]:
FEW_SHOT_EXAMPLES = [
    # 1. Clear NO_CRIME_FRAME — integration context, no security link
    {
        "text": "Die Lage in der Unterkunft hat sich nach Angaben der Sozialarbeiter entspannt. Viele [Gruppe 1] haetten inzwischen Deutschkurse begonnen und bemuehten sich aktiv um eine Arbeitsstelle. Die Caritas lobte das Engagement der Freiwilligen, die regelmaessig Unterstuetzung anbieten.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Der Absatz beschreibt Integration und ehrenamtliche Hilfe ohne jeden Bezug zu Kriminalitaet, Sicherheit oder Praevention."
    },
    # 2. Clear CRIME_FRAME_NEG — direct perpetrator
    {
        "text": "Ein 34-jaehriger [Gruppe 1] wurde am Freitagabend am Hauptbahnhof festgenommen. Er soll in den vergangenen Wochen mehrfach Passanten mit einem Messer bedroht und Handtaschen entrissen haben. Die Polizei hat ihn als Serientaeter eingestuft.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Ein Gruppenmitglied wird explizit als Taeter schwerer Straftaten dargestellt und direkt mit Kriminalitaet verknuepft."
    },
    # 3. CRIME_FRAME_NEG — institutional security framing (fixes missed cases)
    {
        "text": "Die Innenministerkonferenz hat beschlossen, die erkennungsdienstliche Erfassung aller einreisenden [Gruppe 1] zu verschaerfen. Innenminister Hoffmann erklaerte, man koenne sich keine Sicherheitsluecken leisten. Es muesse sichergestellt werden, dass keine Schleuser, Kriminellen oder Extremisten die Situation ausnutzten.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Auch ohne konkrete Straftat wird die Gruppe institutionell als potenzielle Sicherheitsgefahr gerahmt — die explizite Verknuepfung mit Kriminalitaet und Extremismus durch eine Regierungsstimme ist CRIME_FRAME_NEG."
    },
    # 4. NO_CRIME_FRAME — group is victim of violence
    {
        "text": "Auf dem Gelaende der Fluechtlingsunterkunft in Tempelhof wurden in der Nacht Fensterscheiben eingeworfen und Hakenkreuze gesprueht. Die Polizei ermittelt wegen schwerer Sachbeschaedigung und Volksverhetzung. Die [Gruppe 1] Bewohner wurden unverletzt, aber stark veraengstigt in eine Notunterkunft gebracht.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Die [Gruppe 1] sind Opfer eines rechtsextremen Angriffs — eine Gruppe als Opfer von Gewalt darzustellen ist kein Sicherheitsrahmen gegen diese Gruppe."
    },
    # location not mentioned and no Germany link = NO_CRIME_FRAME
    {
        "text": "Berichte zufolge wurden mehrere [Gruppe 1] bei Auseinandersetzungen verletzt. Wo genau die Vorfaelle stattfanden, blieb unklar. Ein Bezug zu Deutschland oder deutschen Sicherheitsbehoerden wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kein expliziter Bezug zu Deutschland oder zur deutschen inneren Sicherheit — ohne diesen Bezug gilt NO_CRIME_FRAME."
    },
    # 5. NO_CRIME_FRAME — international war reporting between states
    {
        "text": "Die [Gruppe 1] Streitkraefte haben nach Angaben des Verteidigungsministeriums in Kiew erneut mehrere Staedte im Osten beschossen. Bei den Angriffen kamen nach ukrainischen Angaben mindestens zwoelf Zivilisten ums Leben. Die NATO beriet ueber weitere Waffenlieferungen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kriegsberichterstattung ueber einen Konflikt zwischen Staaten ohne Bezug zu Minderheiten in Deutschland erhaelt immer NO_CRIME_FRAME."
    },
    # Germany-link exception — foreign crime but explicit Germany connection = CRIME_FRAME_NEG
    {
        "text": "Der Attentaeter, der in Wien drei Menschen toetete, hatte mehrere Jahre in Deutschland gelebt und war deutschen Behoerden als Gefaehrder bekannt. Das Bundesinnenministerium prueft nun, ob Sicherheitsluecken im deutschen System vorlagen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Obwohl die Tat in Oesterreich stattfand, wird ein expliziter Bezug zur deutschen Sicherheit hergestellt — der Taeter lebte in Deutschland und deutsche Behoerden sind beteiligt. Das ergibt CRIME_FRAME_NEG."
    },
    # 6. CRIME_FRAME_NEG — group legitimizes terrorism without being direct perpetrator
    {
        "text": "Sprecher der [Gruppe 1] bezeichneten den Anschlag als legitimen Widerstand und erklaerten, die Opfer seien selbst schuld. In sozialen Medien verbreiteten Anhaenger der Gruppe Videos, in denen der Terrorakt gefeiert wurde.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe legitimiert und feiert explizit einen Terroranschlag — das zaehlt als CRIME_FRAME_NEG auch ohne direkte Taeterschaft."
    },
    # 7. CRIME_FRAME_POS — successful security operation
    {
        "text": "Das Gemeinsame Terrorismusabwehrzentrum hat nach intensiver Zusammenarbeit von Polizei und Geheimdiensten einen islamistischen Anschlag in Muenchen vereitelt. Drei Verdaechtige wurden festgenommen. Innenministerin Weber sprach von einem grossen Erfolg der Sicherheitsbehoerden.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Die Sicherheitsbehoerden werden explizit als erfolgreiche Akteure gerahmt die eine Bedrohung abgewendet haben — Schutz und Sicherheitsgewinn stehen im Vordergrund."
    },
    # 8. CRIME_FRAME_POS — social factors explanation + differentiation (aligns with Robin's definition)
    {
        "text": "Experten erklaeren den Anstieg der Jugendkriminalitaet in Brennpunktvierteln vor allem mit sozialer Ausgrenzung und fehlenden Perspektiven fuer [Gruppe 1] Jugendliche. Kriminologin Mueller betonte, man duerfe keinen Generalverdacht gegen die Gruppe hegen — die grosse Mehrheit halte sich an Gesetze.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Kriminalitaet wird auf soziale Ursachen zurueckgefuehrt und eine explizite Differenzierung wird vorgenommen — das ist CRIME_FRAME_POS nach der Praventions- und Differenzierungsdefinition."
    },
    # 9. CRIME_FRAME_POS — prevention/deradicalization program
    {
        "text": "Das neue Deradikalisierungsprogramm der Bundesregierung soll vor allem [Gruppe 1] Jugendliche ansprechen, die gefaehrdet sind, in extremistische Kreise abzudriften. Sozialarbeiter begleiten die Teilnehmer ueber zwei Jahre und helfen beim Einstieg in Ausbildung und Arbeit.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Ein explizites Praeventionsprogramm im Sicherheitskontext — Schutz und Praevention stehen im Vordergrund, nicht Taeterschaft."
    },
    # 10. NO_CRIME_FRAME — integration without security frame (boundary with CRIME_FRAME_POS)
    {
        "text": "Die Stadtratsfraktion diskutierte Massnahmen zur besseren Integration von [Gruppe 1]. Themen waren Sprachkurse, Wohnraumvermittlung und Beschaeftigungsfoerderung. Ein konkreter Sicherheits- oder Kriminalitaetsbezug wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Integration ohne jeden Sicherheits- oder Kriminalitaetsbezug ist NO_CRIME_FRAME — CRIME_FRAME_POS erfordert einen expliziten Sicherheits- oder Praeventionsrahmen."
    },
    # fixes id 14, 21, 135 — generalised threat/disorder framing
    {
        "text": "Die Behörden berichten, dass sich in Teilen der Stadt regelrechte Parallelgesellschaften gebildet haben, in denen [Gruppe 1] unter sich bleiben und staatliche Kontrolle kaum noch greift. Anwohner klagen über Verwahrlosung und mangelnde Sicherheit.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird mit Kontrollverlust, Unordnung und mangelnder Sicherheit im deutschen Inland verknüpft — das ist generalisierte gesellschaftliche Bedrohungsrahmung und zählt als CRIME_FRAME_NEG."
    },

    # fixes id 30 — explicit domestic security risk framing
    {
        "text": "Der Botschafter warnte: 'Diese [Gruppe 1] stellen ein Sicherheitsrisiko für Deutschland dar.' Es sei falsch, sie aufzunehmen, da sie in deutschen Städten für Unruhe sorgen würden.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird von einer öffentlichen Person explizit als Sicherheitsrisiko für Deutschland bezeichnet — das ist CRIME_FRAME_NEG unabhängig davon ob ein konkretes Verbrechen genannt wird."
    },
    # fixes id 15, 47, 113, 123, 189 — political commentary overcalling
    {
        "text": "Die Politikerin forderte eine bessere Integrationspolitik für [Gruppe 1]. In der Debatte wurden auch Fragen der öffentlichen Ordnung angesprochen, ohne jedoch konkrete Vorfälle oder Straftaten zu nennen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Allgemeine politische Debatten über Integration oder öffentliche Ordnung ohne explizite Verknüpfung der Gruppe mit konkreten Straftaten, Verdacht oder Ermittlungen sind NO_CRIME_FRAME."
    }
]

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    print(f"  {i+1}. {ex['label']}")

Few-shot examples loaded: 15
  1. NO_CRIME_FRAME
  2. CRIME_FRAME_NEG
  3. CRIME_FRAME_NEG
  4. NO_CRIME_FRAME
  5. NO_CRIME_FRAME
  6. NO_CRIME_FRAME
  7. CRIME_FRAME_NEG
  8. CRIME_FRAME_NEG
  9. CRIME_FRAME_POS
  10. CRIME_FRAME_POS
  11. CRIME_FRAME_POS
  12. NO_CRIME_FRAME
  13. CRIME_FRAME_NEG
  14. CRIME_FRAME_NEG
  15. NO_CRIME_FRAME


In [5]:
def call_api(messages, temperature=0.0):
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload, headers=headers, timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def format_few_shot_block(examples):
    blocks = []
    for ex in examples:
        block = f"Text:\n{ex['text']}\n\nLabel: {ex['label']}\nExplanation: {ex['explanation']}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def annotate_few_shot(text, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele fuer die Annotation:\n\n{few_shot_block}\n\n---\n\n"
        f"Jetzt annotieren Sie diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Antworten Sie immer im angegebenen Format."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


print("Functions loaded.")

Functions loaded.


## Load Remaining 250 Rows

In [35]:
original  = pd.read_csv(RESULTS_DIR / "step3_testset_500_for_human_annotation.csv")
completed = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed_translated.csv")

remaining = original[
    ~original["testset_id"].isin(completed["testset_id"])
].copy().reset_index(drop=True)

print(f"Remaining rows: {len(remaining)}")
print(f"testset_id range: {remaining['testset_id'].min()} to {remaining['testset_id'].max()}")
print(remaining["group"].value_counts().head(10))

Remaining rows: 250
testset_id range: 251 to 500
group
RUS       37
REFUG     31
UKR       25
MIGR      19
GBR       13
USA        9
MUSL       9
ASYL       8
ITA        8
ISLMST     7
Name: count, dtype: int64


## Annotate Remaining 250 with Condition B

In [36]:
OUTPUT_NEW = RESULTS_DIR / "step4_new250_model_labels.csv"

if OUTPUT_NEW.exists():
    done = pd.read_csv(OUTPUT_NEW)
    done_ids = set(done["testset_id"])
    print(f"Resuming: {len(done)} done, {len(remaining) - len(done)} remaining")
else:
    done = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(remaining)} rows")

buffer = []

for i, row in remaining.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw         = annotate_few_shot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)

    buffer.append({
        "testset_id":    row["testset_id"],
        "group":         row["group"],
        "text_block_en": row["text_block_en"],
        "model_label":   label,
        "model_explanation": explanation,
    })
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_NEW, mode="a", header=not OUTPUT_NEW.exists(), index=False)
        done  = pd.concat([done, chunk], ignore_index=True)
        buffer = []
        neg = (done["model_label"] == "CRIME_FRAME_NEG").sum()
        pos = (done["model_label"] == "CRIME_FRAME_POS").sum()
        print(f"  [{len(done)}/{len(remaining)}]  NEG: {neg}  POS: {pos}")

if buffer:
    chunk = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_NEW, mode="a", header=not OUTPUT_NEW.exists(), index=False)
    done = pd.concat([done, chunk], ignore_index=True)

print(f"\nDone. {len(done)} rows annotated.")
print(done["model_label"].value_counts())

Resuming: 250 done, 0 remaining

Done. 250 rows annotated.
model_label
NO_CRIME_FRAME     190
CRIME_FRAME_NEG     60
Name: count, dtype: int64


In [11]:
# Step 3.5b alternative: Run fresh model labels for completed new 250

OUTPUT_NEW = RESULTS_DIR / "human_completed_ground_truth_new250_with_fresh_model.csv"
INPUT_NEW = RESULTS_DIR / "human_completed_ground_truth_new250.csv"

new250_completed = pd.read_csv(INPUT_NEW)
new250_completed.columns = new250_completed.columns.str.strip()

def normalize_testset_id(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype(str).replace("<NA>", np.nan)

new250_completed["testset_id"] = normalize_testset_id(new250_completed["testset_id"])

new250_completed = new250_completed[
    new250_completed["testset_id"].notna() &
    new250_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

if OUTPUT_NEW.exists():
    done = pd.read_csv(OUTPUT_NEW)
    done.columns = done.columns.str.strip()
    done["testset_id"] = normalize_testset_id(done["testset_id"])

    done_ids = set(
        done.loc[
            done["model_label"].isin(VALID_LABELS),
            "testset_id"
        ]
    )

    print(f"Resuming: {len(done_ids)} done, {len(new250_completed) - len(done_ids)} remaining")
else:
    done = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(new250_completed)} rows")

buffer = []

for i, row in new250_completed.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw = annotate_few_shot(str(row["text_block_en"]))
    label = parse_label(raw)
    explanation = parse_explanation(raw)

    row_out = row.to_dict()
    row_out["model_label"] = label
    row_out["model_explanation"] = explanation
    row_out["model_raw_output"] = raw
    row_out["model"] = MODEL if "MODEL" in globals() else None

    buffer.append(row_out)
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk = pd.DataFrame(buffer)
        chunk.to_csv(
            OUTPUT_NEW,
            mode="a",
            header=not OUTPUT_NEW.exists(),
            index=False
        )

        done = pd.concat([done, chunk], ignore_index=True)
        buffer = []

        neg = (done["model_label"] == "CRIME_FRAME_NEG").sum()
        pos = (done["model_label"] == "CRIME_FRAME_POS").sum()
        no_frame = (done["model_label"] == "NO_CRIME_FRAME").sum()

        print(
            f"  [{len(done)}/{len(new250_completed)}] "
            f"NEG: {neg}  POS: {pos}  NO_FRAME: {no_frame}"
        )

if buffer:
    chunk = pd.DataFrame(buffer)
    chunk.to_csv(
        OUTPUT_NEW,
        mode="a",
        header=not OUTPUT_NEW.exists(),
        index=False
    )

    done = pd.concat([done, chunk], ignore_index=True)

done = done.drop_duplicates(
    subset=["testset_id"],
    keep="last"
).reset_index(drop=True)

done.to_csv(OUTPUT_NEW, index=False)

print(f"\nDone. {len(done)} rows annotated.")
print(f"Saved to {OUTPUT_NEW}")

print("\nModel label distribution:")
print(done["model_label"].value_counts(dropna=False))

print("\nHuman label distribution:")
print(done["final_human_label"].value_counts(dropna=False))

Resuming: 250 done, 0 remaining

Done. 250 rows annotated.
Saved to results/human_completed_ground_truth_new250_with_fresh_model.csv

Model label distribution:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Human label distribution:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64


## Generate Human Labelling File

Clean file for you and Robin to fill in manually.
Columns: testset_id, group, text_block_en, label_marmee, label_robin, notes.
Model labels and explanations are included so you can use them as a reference,
but fill in your own label independently first before checking the model.

In [37]:
# Re-annotate with updated prompt — saves to separate file
OUTPUT_MODEL = RESULTS_DIR / "step4_new250_model_labels_v2.csv"

labelled_file = pd.read_csv(RESULTS_DIR / "step4_new250_human_labelling_translated.csv")

if OUTPUT_MODEL.exists():
    done_m   = pd.read_csv(OUTPUT_MODEL)
    done_ids = set(done_m["testset_id"])
    print(f"Resuming: {len(done_m)} done, {len(labelled_file) - len(done_m)} remaining")
else:
    done_m   = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(labelled_file)} rows")

buffer = []

for i, row in labelled_file.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw         = annotate_few_shot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)

    buffer.append({
        "testset_id":        row["testset_id"],
        "group":             row["group"],
        "text_block_en":     row["text_block_en"],
        "model_label":       label,
        "model_explanation": explanation,
    })
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk  = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_MODEL, mode="a", header=not OUTPUT_MODEL.exists(), index=False)
        done_m = pd.concat([done_m, chunk], ignore_index=True)
        buffer = []
        print(f"  [{len(done_m)}/{len(labelled_file)}] checkpointed")

if buffer:
    chunk  = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_MODEL, mode="a", header=not OUTPUT_MODEL.exists(), index=False)
    done_m = pd.concat([done_m, chunk], ignore_index=True)

print(f"\nDone. {len(done_m)} rows annotated.")
print(done_m["model_label"].value_counts())

Starting fresh: 250 rows
  [50/250] checkpointed
  [100/250] checkpointed
  [150/250] checkpointed
  [200/250] checkpointed
  [250/250] checkpointed

Done. 250 rows annotated.
model_label
NO_CRIME_FRAME     195
CRIME_FRAME_NEG     55
Name: count, dtype: int64


In [38]:
done = pd.read_csv(OUTPUT_NEW)

# merge with original to get all original columns back
human_file = remaining.merge(
    done[["testset_id", "model_label", "model_explanation"]],
    on="testset_id",
    how="left"
)

# add empty columns for human labels
human_file["label_marmee"] = ""
human_file["label_robin"]  = ""
human_file["notes"]        = ""

# reorder — put text and human label columns first
cols = [
    "testset_id", "group",
    "text_block_en",
    "label_marmee", "label_robin", "notes",
    "model_label", "model_explanation",
]
# add any remaining original columns at the end
extra = [c for c in human_file.columns if c not in cols]
human_file = human_file[cols + extra]

output_human = RESULTS_DIR / "step4_new250_human_labelling.csv"
if not output_human.exists():
    human_file.to_csv(output_human, index=False)
    print(f"Created: {output_human}")
else:
    print(f"File already exists — skipping to preserve existing labels.")

print(f"Human labelling file saved: {output_human}")
print(f"Rows: {len(human_file)}")
print(f"\nModel label distribution (for reference):")
print(human_file["model_label"].value_counts())
print(f"\nGroup distribution:")
print(human_file["group"].value_counts().head(15))

File already exists — skipping to preserve existing labels.
Human labelling file saved: results/step4_new250_human_labelling.csv
Rows: 250

Model label distribution (for reference):
model_label
NO_CRIME_FRAME     190
CRIME_FRAME_NEG     60
Name: count, dtype: int64

Group distribution:
group
RUS       37
REFUG     31
UKR       25
MIGR      19
GBR       13
USA        9
MUSL       9
ASYL       8
ITA        8
ISLMST     7
FRA        6
HUN        6
JUDE       5
FOR        5
JPN        4
Name: count, dtype: int64


## Evaluate Against Human Labels

Run this cell AFTER you and Robin have filled in label_marmee and label_robin.

In [39]:
RESULTS_DIR = Path("results")
df = pd.read_csv(RESULTS_DIR / "step4_new250_human_labelling.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print("label_marmee values:", df["label_marmee"].value_counts(dropna=False).head(10))
print()
print("label_robin values:", df["label_robin"].value_counts(dropna=False).head(10))

Shape: (250, 12)
Columns: ['testset_id', 'group', 'text_block_en', 'label_marmee', 'label_robin', 'notes', 'model_label', 'model_explanation', 'article_id', 'par_index', 'notes_robin', 'notes_marmee']

label_marmee values: label_marmee
NO_CRIME_FRAME     212
CRIME_FRAME_NEG     38
Name: count, dtype: int64

label_robin values: label_robin
NO_CRIME_FRAME     209
CRIME_FRAME_NEG     38
CRIME_FRAME_POS      2
NO_CRIME_FRAME       1
Name: count, dtype: int64


In [40]:
labelled = pd.read_csv(RESULTS_DIR / "step4_new250_human_labelling.csv")

# keep only rows where both annotators filled in a valid label
labelled = labelled[
    labelled["label_marmee"].isin(VALID_LABELS) &
    labelled["label_robin"].isin(VALID_LABELS)
].copy()

print(f"Rows with both human labels: {len(labelled)}")

# consensus: agree = use that label, disagree = UNSURE
labelled["consensus"] = labelled.apply(
    lambda r: r["label_marmee"] if r["label_marmee"] == r["label_robin"] else "UNSURE",
    axis=1
)

agreed = labelled[labelled["consensus"] != "UNSURE"].copy()
unsure = labelled[labelled["consensus"] == "UNSURE"].copy()

print(f"\nAgreed:  {len(agreed)} ({len(agreed)/len(labelled)*100:.1f}%)")
print(f"UNSURE:  {len(unsure)} ({len(unsure)/len(labelled)*100:.1f}%)")

if len(unsure) > 0:
    print(f"\nTestset IDs to review manually:")
    print(unsure["testset_id"].tolist())
    print()
    print("Details:")
    for _, row in unsure.iterrows():
        print(f"\n  id:{row['testset_id']} | group:{row['group']}")
        print(f"  Marmee: {row['label_marmee']} | Robin: {row['label_robin']}")
        print(f"  Text: {str(row['text_block_en'])[:300]}")

# human agreement metrics
kappa_human = cohen_kappa_score(
    labelled["label_marmee"], labelled["label_robin"],
    labels=VALID_LABELS
)
alpha_data = np.array([
    [VALID_LABELS.index(l) for l in labelled["label_marmee"]],
    [VALID_LABELS.index(l) for l in labelled["label_robin"]]
])
alpha_human = krippendorff.alpha(alpha_data, level_of_measurement="nominal")

print(f"\nHuman-Human Agreement:")
print(f"  Cohen's kappa:         {kappa_human:.3f}")
print(f"  Krippendorff's alpha:  {alpha_human:.3f}")
print(f"\nConsensus label distribution:")
print(agreed["consensus"].value_counts())

Rows with both human labels: 249

Agreed:  234 (94.0%)
UNSURE:  15 (6.0%)

Testset IDs to review manually:
[252, 255, 267, 277, 279, 283, 284, 292, 300, 306, 332, 372, 446, 454, 493]

Details:

  id:252 | group:BIH
  Marmee: CRIME_FRAME_NEG | Robin: NO_CRIME_FRAME
  Text: Damals begann die Kolonialisierung Deutschlands, die sich 2015 mit der Grenzöffnung und dem Migrationspakt inzwischen zu einer alles verschlingenden Einwanderungswelle entwickelt hat. Auch in Deutschland wird mit Traumatisierungen zur Zerrüttung und leichteren Kontrolle der autochthonen Einwohner ge

  id:255 | group:SALAF
  Marmee: NO_CRIME_FRAME | Robin: CRIME_FRAME_NEG
  Text: Zwei weitere Schüler zeigen das unter [Andere Gruppe] verbreitete „Rabia“-Zeichen. Entstanden ist der Gruß 2013 in Kairo, als die ägyptische Armee eine Blockade der islamistischen Muslimbruderschaft auflöste. Der türkische Staatschef Recep Tayyip Erdogan verwendete den Gruß, als er im Sommer 2016 na

  id:267 | group:REFUG
  Marmee: CRIME_FRA

In [41]:
# model_label already exists in the file, no need to merge
clean = agreed.copy()

# if model_label is missing for some rows, fill from model results file
if clean["model_label"].isna().any():
    model_results = pd.read_csv(RESULTS_DIR / "step4_new250_model_labels_v2.csv")
    clean = agreed.merge(
        model_results[["testset_id", "model_label", "model_explanation"]],
        on="testset_id",
        how="left"
    )

y_true = clean["consensus"].tolist()
y_pred = clean["model_label"].tolist()

valid  = [(t, p) for t, p in zip(y_true, y_pred) if t in VALID_LABELS and p in VALID_LABELS]
yt, yp = zip(*valid)

acc      = accuracy_score(yt, yp)
f1_macro = f1_score(yt, yp, average="macro", labels=VALID_LABELS, zero_division=0)
f1_w     = f1_score(yt, yp, average="weighted", labels=VALID_LABELS, zero_division=0)
kappa    = cohen_kappa_score(yt, yp, labels=VALID_LABELS)
alpha_data = np.array([
    [VALID_LABELS.index(t) for t in yt],
    [VALID_LABELS.index(p) for p in yp]
])
alpha = krippendorff.alpha(alpha_data, level_of_measurement="nominal")

print(f"\n{'='*50}")
print(f"Model vs Human Consensus (fresh 250)")
print(f"{'='*50}")
print(f"Rows evaluated:        {len(yt)}")
print(f"Accuracy:              {acc:.3f}")
print(f"Macro F1:              {f1_macro:.3f}")
print(f"Weighted F1:           {f1_w:.3f}")
print(f"Cohen's kappa:         {kappa:.3f}")
print(f"Krippendorff's alpha:  {alpha:.3f}")
print()
print(classification_report(yt, yp, labels=VALID_LABELS, zero_division=0))
print("Confusion matrix (rows=human, cols=model):")
print(confusion_matrix(yt, yp, labels=VALID_LABELS))
print(f"Label order: {VALID_LABELS}")

print(f"\n=== Comparison with first 250 ===")
print(f"First 250 kappa (condition B):  0.588")
print(f"Fresh 250 kappa:                {kappa:.3f}")
if kappa >= 0.55:
    print("Result holds on fresh data.")
else:
    print("Kappa dropped — check for overfitting.")


Model vs Human Consensus (fresh 250)
Rows evaluated:        234
Accuracy:              0.902
Macro F1:              0.554
Weighted F1:           0.911
Cohen's kappa:         0.668
Krippendorff's alpha:  0.664

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.99      0.89      0.94       203
CRIME_FRAME_NEG       0.58      0.97      0.72        31
CRIME_FRAME_POS       0.00      0.00      0.00         0

       accuracy                           0.90       234
      macro avg       0.52      0.62      0.55       234
   weighted avg       0.94      0.90      0.91       234

Confusion matrix (rows=human, cols=model):
[[181  22   0]
 [  1  30   0]
 [  0   0   0]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']

=== Comparison with first 250 ===
First 250 kappa (condition B):  0.588
Fresh 250 kappa:                0.668
Result holds on fresh data.


In [12]:
# Model vs final human ground truth for new 250 with fresh model labels

clean = pd.read_csv(RESULTS_DIR / "human_completed_ground_truth_new250_with_fresh_model.csv")
clean.columns = clean.columns.str.strip()


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


required_cols = [
    "final_human_label",
    "model_label"
]

missing_cols = [
    col for col in required_cols
    if col not in clean.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

clean["final_human_label"] = clean_label_series(clean["final_human_label"])
clean["model_label"] = clean_label_series(clean["model_label"])

clean = clean[
    clean["final_human_label"].isin(VALID_LABELS) &
    clean["model_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

if len(clean) == 0:
    raise ValueError("No valid rows found. Check final_human_label and model_label.")

y_true = clean["final_human_label"].tolist()
y_pred = clean["model_label"].tolist()

acc = accuracy_score(y_true, y_pred)

f1_macro = f1_score(
    y_true,
    y_pred,
    average="macro",
    labels=VALID_LABELS,
    zero_division=0
)

f1_w = f1_score(
    y_true,
    y_pred,
    average="weighted",
    labels=VALID_LABELS,
    zero_division=0
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=VALID_LABELS
)

alpha_data = np.array([
    [VALID_LABELS.index(t) for t in y_true],
    [VALID_LABELS.index(p) for p in y_pred]
])

alpha = krippendorff.alpha(
    reliability_data=alpha_data,
    level_of_measurement="nominal"
)

print(f"\n{'=' * 50}")
print("Model vs Final Human Ground Truth, new 250 with fresh model")
print(f"{'=' * 50}")

print(f"Rows evaluated:        {len(clean)}")
print(f"Accuracy:              {acc:.3f}")
print(f"Macro F1:              {f1_macro:.3f}")
print(f"Weighted F1:           {f1_w:.3f}")
print(f"Cohen's kappa:         {kappa:.3f}")
print(f"Krippendorff's alpha:  {alpha:.3f}")

print("\nHuman label distribution:")
print(clean["final_human_label"].value_counts(dropna=False))

print("\nModel label distribution:")
print(clean["model_label"].value_counts(dropna=False))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        zero_division=0
    )
)

print("\nConfusion matrix, rows are human and columns are model:")
conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=VALID_LABELS
)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[f"human_{label}" for label in VALID_LABELS],
    columns=[f"model_{label}" for label in VALID_LABELS]
)

display(conf_matrix_df)

print(f"Label order: {VALID_LABELS}")

print("\n=== Comparison with first 250 ===")
print("First 250 kappa, condition B:  0.588")
print(f"Fresh 250 kappa:               {kappa:.3f}")

if kappa >= 0.55:
    print("Result holds on fresh data.")
else:
    print("Kappa dropped. Check for overfitting.")


Model vs Final Human Ground Truth, new 250 with fresh model
Rows evaluated:        250
Accuracy:              0.804
Macro F1:              0.492
Weighted F1:           0.831
Cohen's kappa:         0.498
Krippendorff's alpha:  0.474

Human label distribution:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Model label distribution:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Classification report:
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       1.00      0.77      0.87       213
CRIME_FRAME_NEG       0.44      1.00      0.61        37
CRIME_FRAME_POS       0.00      0.00      0.00         0

       accuracy                           0.80       250
      macro avg       0.48      0.59      0.49       250
   weighted avg       0.92      0.80      0.83       250


Confusion matrix, rows are human and columns are model:


,model_NO_CRIME_FRAME,model_CRIME_FRAME_NEG,model_CRIME_FRAME_POS
human_NO_CRIME_FRAME,164,48,1
human_CRIME_FRAME_NEG,0,37,0
human_CRIME_FRAME_POS,0,0,0


Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']

=== Comparison with first 250 ===
First 250 kappa, condition B:  0.588
Fresh 250 kappa:               0.498
Kappa dropped. Check for overfitting.


In [13]:
# Model vs final human ground truth for new 250 with fresh model labels

clean = pd.read_csv(RESULTS_DIR / "human_completed_ground_truth_new250_with_fresh_model.csv")
clean.columns = clean.columns.str.strip()


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


required_cols = [
    "final_human_label",
    "model_label"
]

missing_cols = [
    col for col in required_cols
    if col not in clean.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

clean["final_human_label"] = clean_label_series(clean["final_human_label"])
clean["model_label"] = clean_label_series(clean["model_label"])

print("Human label distribution before filtering:")
print(clean["final_human_label"].value_counts(dropna=False))

print("\nModel label distribution before filtering:")
print(clean["model_label"].value_counts(dropna=False))

# Keep only rows where both human and model labels are valid labels
clean = clean[
    clean["final_human_label"].isin(VALID_LABELS) &
    clean["model_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

if len(clean) == 0:
    raise ValueError("No valid rows found. Check final_human_label and model_label.")

# Use every label that appears in either human labels or model predictions
# This means CRIME_FRAME_POS stays in the evaluation if the model predicted it once
observed_labels = set(clean["final_human_label"]).union(set(clean["model_label"]))

EVAL_LABELS = [
    label for label in VALID_LABELS
    if label in observed_labels
]

print("\nEvaluation labels used:")
print(EVAL_LABELS)

y_true = clean["final_human_label"].tolist()
y_pred = clean["model_label"].tolist()

acc = accuracy_score(y_true, y_pred)

f1_macro = f1_score(
    y_true,
    y_pred,
    average="macro",
    labels=EVAL_LABELS,
    zero_division=0
)

f1_w = f1_score(
    y_true,
    y_pred,
    average="weighted",
    labels=EVAL_LABELS,
    zero_division=0
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)

alpha_data = np.array([
    [EVAL_LABELS.index(t) for t in y_true],
    [EVAL_LABELS.index(p) for p in y_pred]
])

alpha = krippendorff.alpha(
    reliability_data=alpha_data,
    level_of_measurement="nominal"
)

print(f"\n{'=' * 50}")
print("Model vs Final Human Ground Truth, new 250 with fresh model")
print(f"{'=' * 50}")

print(f"Rows evaluated:        {len(clean)}")
print(f"Accuracy:              {acc:.3f}")
print(f"Macro F1:              {f1_macro:.3f}")
print(f"Weighted F1:           {f1_w:.3f}")
print(f"Cohen's kappa:         {kappa:.3f}")
print(f"Krippendorff's alpha:  {alpha:.3f}")

print("\nHuman label distribution:")
print(clean["final_human_label"].value_counts(dropna=False))

print("\nModel label distribution:")
print(clean["model_label"].value_counts(dropna=False))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=EVAL_LABELS,
        zero_division=0
    )
)

print("\nConfusion matrix, rows are human and columns are model:")
conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[f"human_{label}" for label in EVAL_LABELS],
    columns=[f"model_{label}" for label in EVAL_LABELS]
)

display(conf_matrix_df)

print(f"Label order: {EVAL_LABELS}")

print("\n=== Comparison with first 250 ===")
print("First 250 kappa, condition B:  0.588")
print(f"Fresh 250 kappa:               {kappa:.3f}")

if kappa >= 0.55:
    print("Result holds on fresh data.")
else:
    print("Kappa dropped. Check for overfitting or instruction drift.")

Human label distribution before filtering:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Model label distribution before filtering:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Evaluation labels used:
['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']

Model vs Final Human Ground Truth, new 250 with fresh model
Rows evaluated:        250
Accuracy:              0.804
Macro F1:              0.492
Weighted F1:           0.831
Cohen's kappa:         0.498
Krippendorff's alpha:  0.474

Human label distribution:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Model label distribution:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Classification report:
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       1.00      0.77      0.87       213
CRIME_FRAME

,model_NO_CRIME_FRAME,model_CRIME_FRAME_NEG,model_CRIME_FRAME_POS
human_NO_CRIME_FRAME,164,48,1
human_CRIME_FRAME_NEG,0,37,0
human_CRIME_FRAME_POS,0,0,0


Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']

=== Comparison with first 250 ===
First 250 kappa, condition B:  0.588
Fresh 250 kappa:               0.498
Kappa dropped. Check for overfitting or instruction drift.


From Robin: Results under this: got rid of CRIME_FRAME_POS and added rules: 
CRIME_FRAME_NEG
- Gewalt, Straftaten oder Sicherheitsbedrohungen, die außerhalb Deutschlands stattfinden (z.B. Kriegshandlungen, Repression durch ausländische Regierungen, geopolitische Konflikte zwischen Staaten im Ausland), erhalten NO_CRIME_FRAME, wenn kein expliziter Bezug zu Deutschland, zur deutschen inneren Sicherheit oder zu einem europäischen bzw. NATO-Sicherheitskontext besteht, in dem Deutschland ausdrücklich betroffen, beteiligt oder gefährdet ist.
- Straftaten, Sicherheitsbedrohungen, Gewalt oder Kriegshandlungen außerhalb Deutschlands zählen nur dann als CRIME_FRAME_NEG, wenn ein expliziter Bezug zu Deutschland oder zu einem europäischen bzw. NATO-Sicherheitskontext besteht, in dem Deutschland erkennbar betroffen, beteiligt oder mitgefährdet ist. Dies gilt z.B. wenn der Täter in Deutschland lebte, deutsche Behörden beteiligt sind, die Sicherheit Deutschlands bedroht wird, der Vorfall Auswirkungen auf Deutschland hat oder der Text einen europäischen bzw. NATO-Sicherheitsrahmen beschreibt, in dem Deutschland Teil des betroffenen Sicherheitsraums ist.

CRIME_FRAME_POS:
- Abschiebungen, Rückführungen oder Ausweisungen zur Eindämmung von Kriminalität sind nicht als CRIME_FRAME_POS zu labeln, nur weil sie als Mittel gegen Kriminalität dargestellt werden. Wenn die Gruppe oder ein Gruppenmitglied dabei als Täter, Verdächtiger, Beschuldigter oder Sicherheitsrisiko dargestellt wird, gilt CRIME_FRAME_NEG. Wenn es nur um ausländerrechtliche Verwaltung ohne klare Verknüpfung mit Kriminalität oder Sicherheit geht, gilt NO_CRIME_FRAME.

Entscheidungsregel:
- Wenn eine Straftat, Bedrohung oder Kriegshandlung außerhalb Deutschlands stattfindet, aber ein expliziter Bezug zu Deutschland oder zu einem europäischen bzw. NATO-Sicherheitskontext hergestellt wird, in dem Deutschland betroffen, beteiligt oder mitgefährdet ist, kann CRIME_FRAME_NEG vergeben werden. Ohne diesen expliziten Bezug → NO_CRIME_FRAME.
- Abschiebungen, Rückführungen oder Ausweisungen zur Eindämmung von Kriminalität sind nicht automatisch CRIME_FRAME_POS. Wenn die Gruppe oder ein Gruppenmitglied als kriminell, verdächtig oder sicherheitsgefährdend dargestellt wird → CRIME_FRAME_NEG. Wenn kein expliziter Kriminalitäts- oder Sicherheitsrahmen vorliegt → NO_CRIME_FRAME. 

In [14]:
# Model vs final human ground truth for new 250 with fresh model labels
# Fully excluding CRIME_FRAME_POS from the evaluation

clean = pd.read_csv(RESULTS_DIR / "human_completed_ground_truth_new250_with_fresh_model.csv")
clean.columns = clean.columns.str.strip()


def clean_label_series(series):
    return (
        series
        .astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "NaN": np.nan,
            "None": np.nan,
            "NONE": np.nan
        })
    )


required_cols = [
    "final_human_label",
    "model_label"
]

missing_cols = [
    col for col in required_cols
    if col not in clean.columns
]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

clean["final_human_label"] = clean_label_series(clean["final_human_label"])
clean["model_label"] = clean_label_series(clean["model_label"])

print("Human label distribution before filtering:")
print(clean["final_human_label"].value_counts(dropna=False))

print("\nModel label distribution before filtering:")
print(clean["model_label"].value_counts(dropna=False))

# First keep only valid labels
clean = clean[
    clean["final_human_label"].isin(VALID_LABELS) &
    clean["model_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

# Now fully exclude CRIME_FRAME_POS from the evaluation
EVAL_LABELS = [
    label for label in VALID_LABELS
    if label != "CRIME_FRAME_POS"
]

before_excluding_pos = len(clean)

clean_eval = clean[
    clean["final_human_label"].isin(EVAL_LABELS) &
    clean["model_label"].isin(EVAL_LABELS)
].copy().reset_index(drop=True)

rows_excluded_pos = before_excluding_pos - len(clean_eval)

print("\nEvaluation labels used:")
print(EVAL_LABELS)

print(f"\nRows before excluding CRIME_FRAME_POS: {before_excluding_pos}")
print(f"Rows excluded because human or model label was CRIME_FRAME_POS: {rows_excluded_pos}")
print(f"Rows evaluated after excluding CRIME_FRAME_POS: {len(clean_eval)}")

if len(clean_eval) == 0:
    raise ValueError("No valid rows found after excluding CRIME_FRAME_POS.")

y_true = clean_eval["final_human_label"].tolist()
y_pred = clean_eval["model_label"].tolist()

acc = accuracy_score(y_true, y_pred)

f1_macro = f1_score(
    y_true,
    y_pred,
    average="macro",
    labels=EVAL_LABELS,
    zero_division=0
)

f1_w = f1_score(
    y_true,
    y_pred,
    average="weighted",
    labels=EVAL_LABELS,
    zero_division=0
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)

alpha_data = np.array([
    [EVAL_LABELS.index(t) for t in y_true],
    [EVAL_LABELS.index(p) for p in y_pred]
])

alpha = krippendorff.alpha(
    reliability_data=alpha_data,
    level_of_measurement="nominal"
)

print(f"\n{'=' * 50}")
print("Model vs Final Human Ground Truth, new 250, CRIME_FRAME_POS excluded")
print(f"{'=' * 50}")

print(f"Rows evaluated:        {len(clean_eval)}")
print(f"Accuracy:              {acc:.3f}")
print(f"Macro F1:              {f1_macro:.3f}")
print(f"Weighted F1:           {f1_w:.3f}")
print(f"Cohen's kappa:         {kappa:.3f}")
print(f"Krippendorff's alpha:  {alpha:.3f}")

print("\nHuman label distribution:")
print(clean_eval["final_human_label"].value_counts(dropna=False))

print("\nModel label distribution:")
print(clean_eval["model_label"].value_counts(dropna=False))

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=EVAL_LABELS,
        zero_division=0
    )
)

print("\nConfusion matrix, rows are human and columns are model:")
conf_matrix = confusion_matrix(
    y_true,
    y_pred,
    labels=EVAL_LABELS
)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=[f"human_{label}" for label in EVAL_LABELS],
    columns=[f"model_{label}" for label in EVAL_LABELS]
)

display(conf_matrix_df)

print(f"Label order: {EVAL_LABELS}")

print("\n=== Comparison with first 250 ===")
print("First 250 kappa, condition B:  0.588")
print(f"Fresh 250 kappa:               {kappa:.3f}")

if kappa >= 0.55:
    print("Result holds on fresh data.")
else:
    print("Kappa dropped. Check for overfitting or instruction drift.")

Human label distribution before filtering:
final_human_label
NO_CRIME_FRAME     213
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Model label distribution before filtering:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
CRIME_FRAME_POS      1
Name: count, dtype: int64

Evaluation labels used:
['NO_CRIME_FRAME', 'CRIME_FRAME_NEG']

Rows before excluding CRIME_FRAME_POS: 250
Rows excluded because human or model label was CRIME_FRAME_POS: 1
Rows evaluated after excluding CRIME_FRAME_POS: 249

Model vs Final Human Ground Truth, new 250, CRIME_FRAME_POS excluded
Rows evaluated:        249
Accuracy:              0.807
Macro F1:              0.739
Weighted F1:           0.833
Cohen's kappa:         0.504
Krippendorff's alpha:  0.480

Human label distribution:
final_human_label
NO_CRIME_FRAME     212
CRIME_FRAME_NEG     37
Name: count, dtype: int64

Model label distribution:
model_label
NO_CRIME_FRAME     164
CRIME_FRAME_NEG     85
Name: count, dtype: int64

Classification repor

,model_NO_CRIME_FRAME,model_CRIME_FRAME_NEG
human_NO_CRIME_FRAME,164,48
human_CRIME_FRAME_NEG,0,37


Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG']

=== Comparison with first 250 ===
First 250 kappa, condition B:  0.588
Fresh 250 kappa:               0.504
Kappa dropped. Check for overfitting or instruction drift.
